# Introducción a NumPy

En los módulos anteriores trabajamos con números, strings, listas, diccionarios, archivos de texto y registros biológicos. Con eso ya podemos resolver muchos problemas reales. NumPy agrega una herramienta nueva: los **arrays**, una estructura pensada para trabajar con muchos valores numéricos al mismo tiempo.

La idea de esta clase no es estudiar matemática avanzada ni todos los detalles internos de NumPy. Vamos a usarlo como una herramienta práctica para hacer cálculos sobre datos biológicos, comparando lo que ya sabemos hacer con listas y loops con una forma más directa de trabajar con vectores.

## Cargar la biblioteca

NumPy suele importarse con el alias `np`. Si esta celda da un error del tipo `No module named 'numpy'`, hay que instalar la biblioteca antes de continuar.

In [ ]:
import numpy as np

## De listas a arrays

Empecemos con un ejemplo que ya apareció en una clase anterior: una lista con largos de genes de *Saccharomyces cerevisiae*. Hasta ahora, si queríamos transformar esos valores o calcular algo con todos ellos, usábamos un loop `for`.

In [ ]:
largos_genes = [453, 915, 186, 1194, 1227, 1980, 942, 882, 1302]
largos_genes

In [ ]:
largos_kb = []

for largo in largos_genes:
    largos_kb.append(largo / 1000)

largos_kb

NumPy permite convertir esa lista en un array. A primera vista se parece a una lista, pero internamente está preparado para hacer operaciones numéricas elemento por elemento.

In [ ]:
largos_array = np.array(largos_genes)
largos_array

In [ ]:
type(largos_array)

In [ ]:
largos_array / 1000

La división anterior se aplicó a todos los elementos del array sin escribir un loop explícito. Esto se conoce como una operación **vectorizada**. Para quienes vienen de R, esta idea debería resultar familiar.

## Algunas operaciones descriptivas

In [ ]:
print(largos_array.min())
print(largos_array.max())
print(largos_array.mean())
print(largos_array.std())

También podemos hacer preguntas que devuelven valores lógicos. En este ejemplo preguntamos qué genes tienen 900 nucleótidos o más.

In [ ]:
largos_array >= 900

In [ ]:
largos_array[largos_array >= 900]

La expresión `largos_array >= 900` devuelve un array de valores `True` y `False`. Cuando usamos ese array entre corchetes, NumPy conserva solamente los elementos que corresponden a `True`. Esta forma de filtrar datos se parece mucho a lo que después vamos a usar con Pandas.

### Para probar

Modifiquen el valor de corte del ejemplo anterior:

* ¿Cuántos genes tienen menos de 500 nucleótidos?
* ¿Cuál es el largo promedio de los genes que tienen 900 nucleótidos o más?
* ¿Qué pasa si escriben `largos_array > largos_array.mean()`?

## Leer columnas numéricas desde un archivo

Ahora vamos a volver al archivo `genoma_levadura_S288C.csv`, que usamos en el módulo de archivos. Aunque el nombre termina en `.csv`, los campos están separados por tabuladores. NumPy puede leer columnas específicas de un archivo de texto con `loadtxt`.

Para este primer ejemplo vamos a recuperar columnas numéricas: tamaño del cromosoma, porcentaje de GC, cantidad de proteínas, cantidad de tRNAs, otros RNAs y cantidad total de genes.

In [ ]:
datos_genoma = np.loadtxt(
    "genoma_levadura_S288C.csv",
    delimiter="\t",
    skiprows=1,
    usecols=(4, 5, 6, 8, 9, 10)
)

datos_genoma

In [ ]:
datos_genoma.shape

El atributo `shape` nos dice cuántas filas y columnas tiene el array. En este caso cada fila corresponde a un cromosoma o al genoma mitocondrial, y cada columna corresponde a una variable numérica.

In [ ]:
tamano_mb = datos_genoma[:, 0]
gc = datos_genoma[:, 1]
proteinas = datos_genoma[:, 2]
trnas = datos_genoma[:, 3]
otros_rnas = datos_genoma[:, 4]
genes = datos_genoma[:, 5]

In [ ]:
print(tamano_mb)
print(gc)
print(genes)

La notación `datos_genoma[:, 0]` quiere decir: todas las filas (`:`) y la columna cero. Es la misma lógica de posiciones que vimos con listas, pero ahora en dos dimensiones: filas y columnas.

## Cálculos usando columnas completas

In [ ]:
genes_por_mb = genes / tamano_mb
genes_por_mb

In [ ]:
print(genes_por_mb.min())
print(genes_por_mb.max())
print(genes_por_mb.mean())

Este cálculo es útil para comparar densidad génica entre cromosomas de tamaños diferentes. El genoma mitocondrial de levadura tiene una composición distinta al genoma nuclear, así que conviene mirarlo con cuidado en los resultados.

In [ ]:
gc_bajo = gc < 30
gc_bajo

In [ ]:
datos_genoma[gc_bajo, :]

En el archivo, el registro con `GC%` bajo corresponde al genoma mitocondrial. NumPy no nos muestra el nombre porque en este ejemplo leímos solamente columnas numéricas. En Pandas vamos a poder mezclar columnas numéricas y de texto de una forma más cómoda.

### Para probar

Usando las variables que acabamos de crear:

* Calculen cuántas proteínas hay por megabase.
* Recuperen las filas con más de 500 genes.
* Comparen el promedio de `GC%` incluyendo y excluyendo el genoma mitocondrial.
* Cambien `usecols` para leer otras columnas numéricas. ¿Qué problema aparece si intentan leer una columna que contiene guiones `-`?

## Un ejemplo con secuencias

NumPy no reemplaza a Biopython para trabajar con secuencias, pero puede servir para contar o transformar datos una vez que los convertimos en números. Vamos a leer el archivo fasta de la mitocondria y calcular la frecuencia de cada nucleótido.

In [ ]:
secuencia = ""

with open("S288c_mitochondrion.fasta", "r") as fasta:
    for linea in fasta:
        if not linea.startswith(">"):
            secuencia = secuencia + linea.strip()

len(secuencia)

In [ ]:
nucleotidos = np.array(list(secuencia))
nucleotidos[0:20]

In [ ]:
for base in ["A", "C", "G", "T"]:
    cantidad = np.sum(nucleotidos == base)
    frecuencia = cantidad / len(nucleotidos)
    print(base, cantidad, frecuencia)

En la línea `np.sum(nucleotidos == base)`, la comparación produce un array de valores `True` y `False`. NumPy puede sumar esos valores porque interpreta `True` como 1 y `False` como 0. Así obtenemos el número de posiciones donde aparece cada base.

### Para probar

Extiendan el ejemplo anterior:

* Calculen el contenido GC como `(G + C) / largo_total`.
* Comparen ese resultado con el valor `GC%` informado para `MT` en `genoma_levadura_S288C.csv`.
* Prueben el mismo procedimiento con el archivo `S288c_mitochondrion_tRNAs.fasta`. ¿Qué cambia si las secuencias tienen `U` en lugar de `T`?

## Repaso

Al llegar aquí deberías entender estos puntos:

* Cómo crear un array de NumPy a partir de una lista.
* Cómo hacer operaciones vectorizadas sin escribir un loop explícito.
* Cómo filtrar un array usando condiciones lógicas.
* Cómo leer columnas numéricas de un archivo de texto.
* Por qué NumPy es útil para datos numéricos y por qué Pandas será más cómodo para tablas con columnas mixtas.